In [2]:
###### IMPORT AND UTILITIES ######

import torch
import librosa as lr
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io.wavfile
from IPython.display import Audio

In [3]:
###### LOADING THE AUDIOS ######

files = [f for f in os.listdir('../data/generated/') if f.endswith('wav')]
files.sort()
print(f'files = {files}\n')

amplitudes = {}
sampling_rates = {}

for i, file in enumerate(files):
    print(f'loading sample {i+1}...')
    file_path = os.path.join('../data/generated/', file)
    amp, sr = lr.load(file_path, sr = None)                
    amplitudes[file] = amp                                 
    sampling_rates[file] = sr                              
                                                           
print('\nSuccessfully loaded')
print(f'sampling rate: {sr}Hz')

files = ['sample_1_a_90s_rock_song_with.wav', 'sample_2_a_calm_relaxing_lofi.wav', 'sample_3_epic_cinematic_music.wav', 'sample_4_a_simple_melancholic.wav']

loading sample 1...
loading sample 2...
loading sample 3...
loading sample 4...

Successfully loaded
sampling rate: 32000Hz


In [9]:
###### DENOISERS LOADING ######

print('loading model "dns64" ...')
dns64 = torch.hub.load('facebookresearch/denoiser', 'dns64', pretrained = True)
dns64.eval()
print('model "dns64" loaded successfully\n')

print('loading model "master64" ...')
master64 = torch.hub.load('facebookresearch/denoiser', 'master64', pretrained = True)
master64.eval()
print('model "master64" loaded successfully')

# torch.hub.load is a pytorch function used to load models from github repositories or other online sources.
# The first argument tells it to go to the user "facebookresearch" and locate the repository "denoiser". Second argument specifies 
# which model we want to load. I'm going to try and compare two different models: 

#    ·dns64: belongs to 'dns' models, specifically optimized for the DNS challenges (Deep Noise Suppression). They're more focused
#            on speech, but the datasets used for training contain a lot of background noise.  

#    ·master64: belongs to 'master' models, trained on a very large dataset that includes music, speech and ambient sounds.
#               They are more generic models, but they are also older.

# Lastly, pretrained = True serves to download the model with his pre-trained weights and model.eval() sets the model to evaluation mode.

loading model "dns64" ...


Using cache found in /Users/azzurragiordano/.cache/torch/hub/facebookresearch_denoiser_main


model "dns64" loaded successfully

loading model "master64" ...
model "master64" loaded successfully


Using cache found in /Users/azzurragiordano/.cache/torch/hub/facebookresearch_denoiser_main


In [ ]:
###### DENOISER FUNCTION ######

def denoiser_apply(model, audio):
    
    '''
    Applies a denoising model to audio.

    input: the model we want to use and generated audio's path 
    output: restored audio's path
    
    '''

    original_sr = sampling_rates[audio]
    original_amp = amplitudes[audio]

    model_sr = 16000    # The model works with sr = 16000Hz

    resampled_amp = librosa.resample(original_amp, orig_sr = original_sr, target_sr = model_sr)

    amp_torch = torch.tensor(resampled_amp).unsqueeze(0).unsqueeze(0) # The model expects a three-dimensional input
                                                                      # [batch_size, channels, size], so we added two dimensions at the
                                                                      # beginning. Now wav_torch's shape is [1, 1, len(resampled_amp)].
    with torch.no_grad(): # Disables gradient computation
        restored_torch = model(amp_torch)[0]

    restored_numpy = restored_torch.squeeze().cpu().numpy()

    restored_amp = librosa.resample(original_amp, orig_sr = model_sr, target_sr = original_sr)

In [10]:
amplitudes

{'sample_1_a_90s_rock_song_with.wav': array([-0.05430177, -0.06345857, -0.02517554, ...,  0.0497744 ,
         0.04719456,  0.05138797], shape=(318080,), dtype=float32),
 'sample_2_a_calm_relaxing_lofi.wav': array([-0.33379006, -0.33486873, -0.33174938, ...,  0.00668278,
         0.00955693,  0.01094951], shape=(318080,), dtype=float32),
 'sample_3_epic_cinematic_music.wav': array([ 0.01469086,  0.01359075,  0.01694086, ..., -0.04042038,
        -0.03888955, -0.03824943], shape=(318080,), dtype=float32),
 'sample_4_a_simple_melancholic.wav': array([-0.01012845, -0.01154613, -0.01130864, ...,  0.014334  ,
         0.0149097 ,  0.01397735], shape=(318080,), dtype=float32)}